In [1]:
policy_text = """Bajaj Finance personal loan interest rate starts at 12% per annum.
Processing fee is 2% of the loan amount. EMI bounce attracts a
Rs.500 penalty charge. Prepayment charges are 4% of the outstanding
principal, waived after 12 EMIs.Eligibility: Age 21-60 years. Minimum income Rs 25,000 per month.
CIBIL score 700 and above for standard approval.
FOIR maximum 50 percent of monthly income.
"""

In [2]:
policy_text[0:50]

'Bajaj Finance personal loan interest rate starts a'

In [3]:
chunks = []
start = 0
chunk_size = 100

while start < len(policy_text):
    end = start + chunk_size
    chunks.append(policy_text[start:end])
    start = end

chunks

len(chunks[0])

100

In [4]:
#policy_text.split()
words = policy_text.split()
chunks = []
current_chunk = []
current_len = 0

for word in words:
    word_len = len(word) + 1 # +1 for the space
    if  current_len + word_len > chunk_size:
        chunks.append(" ".join(current_chunk))
        current_chunk = [word]
        current_len = word_len
else:
    current_chunk.append(word)
    current_len += word_len

if  current_chunk:
    chunks.append(" ".join(current_chunk))


In [5]:
print(chunks)

['income.']


In [6]:
print(policy_text)

Bajaj Finance personal loan interest rate starts at 12% per annum.
Processing fee is 2% of the loan amount. EMI bounce attracts a
Rs.500 penalty charge. Prepayment charges are 4% of the outstanding
principal, waived after 12 EMIs.Eligibility: Age 21-60 years. Minimum income Rs 25,000 per month.
CIBIL score 700 and above for standard approval.
FOIR maximum 50 percent of monthly income.



In [7]:
#overlap
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

splitter = CharacterTextSplitter(
    separator =".",
    chunk_size = 30,
    chunk_overlap=10
)


chunks =splitter.split_text(policy_text)

Created a chunk of size 65, which is longer than the specified 30
Created a chunk of size 40, which is longer than the specified 30
Created a chunk of size 77, which is longer than the specified 30
Created a chunk of size 35, which is longer than the specified 30
Created a chunk of size 48, which is longer than the specified 30
Created a chunk of size 42, which is longer than the specified 30


In [8]:
chunks

['Bajaj Finance personal loan interest rate starts at 12% per annum',
 'Processing fee is 2% of the loan amount',
 'EMI bounce attracts a\nRs',
 '500 penalty charge',
 'Prepayment charges are 4% of the outstanding\nprincipal, waived after 12 EMIs',
 'Eligibility: Age 21-60 years',
 'Minimum income Rs 25,000 per month',
 'CIBIL score 700 and above for standard approval',
 'FOIR maximum 50 percent of monthly income']

In [9]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ".", "," ],
    chunk_size=30,
    chunk_overlap=10
)

chunks = splitter.split_text(policy_text)

len(chunks)

19

In [10]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_path ="bajaj_finance_policy_reference.pdf"
loader = PyPDFLoader(pdf_path)
raw_docs = loader.load()

C:\Users\Deepam\AppData\Local\Temp\ipykernel_20436\258883641.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [11]:
len(raw_docs[0].page_content)

1153

In [12]:
raw_docs

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-11T14:01:35+00:00', 'author': 'Bajaj Finance Limited', 'keywords': '', 'moddate': '2026-05-11T14:01:35+00:00', 'subject': 'Internal Policy Manual FY 2024-25', 'title': 'Bajaj Finance Policy Reference Document', 'trapped': '/False', 'source': 'bajaj_finance_policy_reference.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1'}, page_content='BAJAJ FINANCE LIMITED\nInternal Policy Reference Document — CONFIDENTIAL\nPage 1\nFor internal use only | Bajaj Finance Limited | FY 2024-25 | Unauthorised distribution prohibited\n BAJAJ FINANCE LIMITED\n Policy & Procedure Reference\n Document\n Helpdesk Agent Knowledge Base\n FY 2024–25 | All Loan Products\nDocument Type\nInternal Policy & Procedure Manual\nCoverage\nPersonal Loan · Home Loan · Gold Loan · Business Loan · CIBIL Policy\nIntended Users\nHelpdesk Agents · Branch Executives · Collections Team · Sales Team\nCla

In [13]:
from uuid import uuid4

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

load_dotenv()

True

In [14]:
collection_name = "demo2"

In [15]:
embeddings = OpenAIEmbeddings(
    model = "text-embedding-3-small",
    api_key=os.getenv("OPENAI_API_KEY")
)

In [16]:
vetor_store = Chroma(
    collection_name = collection_name,
    embedding_function=embeddings,
    persist_directory="chroma_db",
   # hnsw_config={"ef_contruction": 200, "m":16},
   # ivf_confog={"nlist":100}

)

In [17]:
document_examples = [
    {
        "topic": "AI",
        "doc_number": 1,
        "text": "Artificial intelligence helps machines perform tasks that usually need human reasoning.",
    },
    {
        "topic": "AI",
        "doc_number": 2,
        "text": "AI systems can analyze patterns in data to support predictions and automation.",
    },
    {
        "topic": "AI",
        "doc_number": 3,
        "text": "Responsible AI development includes fairness, transparency, and safety checks.",
    },
    {
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG combines retrieval with generation so the model can answer using external knowledge.",
    },
    {
        "topic": "RAG",
        "doc_number": 5,
        "text": "A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.",
    },
    {
        "topic": "RAG",
        "doc_number": 6,
        "text": "Vector stores are important in RAG because they make semantic search over embedded documents possible.",
    },
    {
        "topic": "LLM",
        "doc_number": 7,
        "text": "LLMs generate text by predicting likely next tokens from patterns learned during training.",
    },
    {
        "topic": "LLM",
        "doc_number": 8,
        "text": "Prompt design can improve how clearly an LLM follows instructions and returns useful answers.",
    },
    {
        "topic": "Cricket",
        "doc_number": 9,
        "text": "Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.",
    },
    {
        "topic": "Cricket",
        "doc_number": 10,
        "text": "A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.",
    },
]

In [18]:

documents = [
    Document(
        id = str(uuid4()),
        page_content=item["text"],
        meta={"topic": item["topic"], "doc_number": item["doc_number"]}
    )
    for item in document_examples
]

documents

[Document(id='6c0d50f7-4aec-4f1a-a9b5-43ec785d8138', metadata={}, page_content='Artificial intelligence helps machines perform tasks that usually need human reasoning.'),
 Document(id='fe6cb9b4-5646-44e8-8cfa-a3c3d49e57bf', metadata={}, page_content='AI systems can analyze patterns in data to support predictions and automation.'),
 Document(id='0685d1e0-442f-444e-bc14-b6b29a2b55c6', metadata={}, page_content='Responsible AI development includes fairness, transparency, and safety checks.'),
 Document(id='00e4771c-47b5-4da3-9ce7-d9655b8f1208', metadata={}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
 Document(id='c04b0037-3baf-4878-bf57-467f29a947ce', metadata={}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'),
 Document(id='9cc9f9d9-b6ef-4961-902f-f9585954e72b', metadata={}, page_content='Vector stores are important in RAG because they make semantic search ov